# Per-pixel decomposition diagnostics

Run a decomposition, find the pixels the model reconstructs *worst*, and for each of them
open up **why** — three lenses:

1. **Loss landscape** — sweep each of the five material parameters
   (albedo R/G/B, roughness, metallic) one at a time, re-render the pixel under every
   image, and plot the data loss. Prediction and ground truth are marked, so you can see
   whether the fit sits in the true basin, a wrong local minimum, or a flat valley.
2. **Specular response** — the specular term alone over the same sweeps. This is usually
   where roughness/metallic are (un)identifiable: if the curve is flat, the data does not
   constrain that parameter.
3. **Gradient breakdown** — the data-loss gradient at the fitted point, split into the
   part flowing through the **diffuse** path and the part through the **specular** path,
   per parameter and per channel. Shows what the optimizer actually "feels" here.

Everything renders through the project's own `shade_ct_sh`, so the analysis is of exactly
the model that was fitted. Defaults to a self-contained synthetic scene (built with
`shade_ct_sh`, so it has exact GT and always runs); point `SCENE` at a dataset directory
to analyse a real run.


In [ ]:
import os, sys
from pathlib import Path

# Repo root (walk up to the dir containing idr/), plus notebooks/ so `pixel_diag` imports.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "idr").is_dir())
for _p in (str(REPO), str(REPO / "notebooks")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
os.chdir(REPO)
os.environ.setdefault("WANDB_MODE", "disabled")

import numpy as np
import torch
import matplotlib.pyplot as plt

from pixel_diag import PixelDiag, gt_render_floor, compare_floor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("repo:", REPO, "| device:", DEVICE)


## 1. Configuration

`SCENE` is either `"synthetic"` (self-contained, exact GT) or a path to a dataset leaf
(`.../<view>/<dataset>/` holding `light_*.npy`, `sh_*.npy`, and the GT maps).

The decomposition config is an ordinary `cfg` dict — the same one `decompose_scene`
takes — so you can point this at LBFGS, LM, or VARPRO (incl. a `curriculum`).

In [ ]:
# ── what to decompose ─────────────────────────────────────────────────────────
# SCENE      = "synthetic"          # "synthetic"  OR  a dataset-leaf path (string / Path)
SCENE      = r"E:\3D-Front\results\3dfront-batch\datasets\1c349305_v0\ct-ct_sh-frOn_env"          # "synthetic"  OR  a dataset-leaf path (string / Path)
N_IMAGES   = 16                   # observations to fit
DOWNSAMPLE = 8                    # only used for a real dataset (synthetic ignores it)

# ── how to decompose (any cfg decompose_scene accepts) ────────────────────────
from idr.config import DEFAULT_CFG
CFG = {**DEFAULT_CFG,
       "optimizer": "ADAM", "n_iter": 20000, 
       # "lbfgs_max_iter": 40,
       "loss": "L2", "double": False, "sh_order": 2,
       "tr_albedo": "sigmoid", "tr_metallic": "sigmoid", "tr_roughness": "sigmoid",
       "init_roughness_zero": True, 
       # "lambda_tv": 1e-5       
       # e.g. VarPro polish from a GD warm start:
       # "optimizer": "VARPRO", "n_iter": 20, "varpro_space": "natural",
       # "curriculum": [{"optimizer": "LBFGS", "n_iter": 200}],
       }

# ── analysis knobs ────────────────────────────────────────────────────────────
N_WORST  = 4                      # how many worst-recon pixels to dissect
SWEEP_N  = 121                    # samples per parameter sweep
SEED     = 0                      # tie-break / reproducibility


## 2. Load the scene

Both sources are normalised into one structure: GT maps, the masked per-pixel geometry
`shade_ct_sh` consumes (`view`, `normal`, the GGX LUT), the observations `(n, M, 3)`, and
the GT SH lighting per image.

In [ ]:
D = PixelDiag.load(SCENE, CFG, N_IMAGES, DOWNSAMPLE, DEVICE, N_WORST, SWEEP_N)


## 3. Run the decomposition

The recovered SH lighting is what the analysis re-lights with — the material sweeps below
hold lighting fixed at this estimate, exactly as the optimizer saw it.

In [ ]:
D.fit()


## 4. Reconstruction error → worst pixels

`render_px` is the one primitive everything else calls: it re-renders a single pixel
under **all** images for a given `(albedo, roughness, metallic)`, using the fitted
lighting. Because it is `shade_ct_sh`, the sweeps and gradients are of the true model.

Worst pixels are ranked by per-pixel RMSE over images at the fitted material.

In [ ]:
D.show_worst()


## 5. Loss landscape per parameter

For each worst pixel and each of the five parameters, sweep that parameter across its
range with the other four held at the fitted value, and plot the **total data loss**
(summed squared residual over all images) at that pixel.

Read it as: is the fit (dashed) at the bottom of the true basin (solid = GT)? A flat
curve means the data does not constrain that parameter here; two minima mean the pixel is
ambiguous; a fit away from a sharp GT minimum means the optimizer stalled.

> These sweeps hold the **lighting fixed** at the fit. Whether the pixel can *actually*
> improve also depends on the shared lighting — see section 8.

In [ ]:
D.show_landscape("loss")


## 6. Specular term over the parameters

The same sweeps, but plotting only the **specular** contribution (mean `|spec|` over
images). This isolates where roughness and metallic act: the diffuse term barely moves
with them, so a flat specular curve means that parameter is unobservable at this pixel —
the loss landscape above will be flat there too, and the fit is then determined by the
prior/init rather than the data.

In [ ]:
D.show_landscape("spec")


## 7. Gradient breakdown

At the fitted point, decompose the data-loss gradient. The loss is
`sum_k ||diff_k + spec_k - obs_k||^2`, and since the render is the *sum* of a diffuse and
a specular term, the gradient splits **additively** by path:

`d loss/d theta = sum_k 2 r_k · (d diff_k/d theta + d spec_k/d theta)`

with `r_k` the residual. Freezing `r_k` and back-propagating through the diffuse-only and
specular-only renders gives the two contributions exactly. This says *how the optimizer's
signal reaches each parameter*: a parameter whose gradient arrives almost entirely through
the specular path is only weakly identified whenever the specular term is small.

In [ ]:
D.show_grad_breakdown()


### Reading the breakdown

- **Total near zero, both paths near zero** — converged and identifiable: the residual is
  small and there is no signal left to move on.
- **Total near zero, but the two paths are large and cancel** — a trade-off direction
  (classically albedo↔lighting scale, or roughness↔metallic through the specular lobe).
  The fit is at a saddle in that pair; the loss landscape shows the flat valley.
- **Gradient arrives almost entirely through the specular path** — that parameter is only
  as well-determined as the specular term is large. Cross-check with §6: if the specular
  magnitude is flat over that parameter, it is effectively unconstrained and the fit
  reflects the init/prior, not the data.


## 8. Why is the pixel stuck? — bringing the lighting in

Sections 5-7 hold the lighting fixed at the fit, so they only answer *"is the material at
its optimum given this lighting"*. But lighting is jointly optimised and **shared across
every pixel**, so a pixel can be stuck for reasons material-only views cannot see. Three
questions separate the causes.

**Can the model even represent this pixel?** The GT lighting is known (it is what the
scene was rendered under), so re-render with each combination of {fitted, GT} material x
{fitted, GT} lighting:

- `r(GT, GT)` is the **model floor**. If it is large, no material or lighting fits this
  pixel — a cast shadow, interreflection, or non-Cook-Torrance BRDF the SH model cannot
  express. No optimiser will fix it.
- fitted-on-diagonal small but both off-diagonal swaps large means the fitted material and
  fitted lighting are **jointly consistent but individually wrong** — a degeneracy (the
  albedo<->lighting split is ambiguous), not an error the loss can feel.


In [ ]:
D.show_residual_attribution()


**Does the pixel still *want* the lighting to move — but can't get it?**
Lighting is one set of SH coefficients shared by every pixel, and at convergence the
*global* lighting gradient is ~0 (it is a free variable the optimiser has already settled).
So the shared lighting is at rest. If an individual pixel's own lighting gradient is still
large, that pixel is demanding a lighting change the consensus will not grant: any move
that helps it raises the loss on the majority by as much, so the optimiser holds still and
the pixel stays stuck. (A cosine against the global gradient is *not* used here: at
convergence that global vector is ~0, so its direction is numerical noise.)

Compare each pixel's lighting-gradient norm to the global one, `|G|/n_px`, the residual
average force. A pixel far above that line has unmet lighting demand — it is a
shared-lighting compromise loser.

In [ ]:
D.show_demand()


### Putting it together — a decision procedure for "why is it stuck"

1. **`r(GT, GT)` large** (residual-attribution top-left) -> the CT+SH model cannot
   represent this pixel. It is not an optimiser failure; the residual is irreducible.
2. **Model floor ~0, but the fit's albedo is far from GT and `r(fit, GT-light)` is large**
   -> a **degeneracy**: fitted material and fitted lighting are a different, equally-valid
   joint explanation. The loss is flat along the albedo<->lighting-scale direction, so
   there is no gradient to pull the split toward GT.
3. **Material gradient small but the pixel's lighting gradient is large vs `|G|/n_px`** ->
   its material is about as good as it gets for the shared lighting, and that lighting is
   pinned by the majority (global light force ~0). The pixel cannot improve without a
   lighting move the rest of the scene forbids.
4. **A parameter's fit sits at a box edge in section 5** (e.g. albedo pressed to 0 on an
   under-lit pixel) -> it is at a constraint; the unconstrained gradient may be non-zero
   but the projected step is zero.

Cases 2-4 are *convergence*, not failure: the optimiser is at a genuine stationary point
of the joint problem. Only case 1 is a modelling limit. The distinction matters — more
iterations help none of them, but case 1 needs a richer model, case 2 needs a prior or
scale gauge, and case 3 needs down-weighting the majority (or per-pixel lighting).

## 9. The two basins — a joint fit→GT loss slice

Sections 5–7 sweep **one material axis with the lighting pinned at the fit**, so they can
only climb *out* of the fit's own basin — monotone, featureless. But the GT solution
differs in material **and** lighting simultaneously, so it lies on no single axis. The way
to see the real landscape is to interpolate *both* fit→GT at once: material along x,
lighting along y, with the fit at `(0,0)` and GT at `(1,1)`.

The **edges** of the resulting surface are exactly the 1D sweeps you already plotted (one
coordinate moving, the other held at the fit). The **diagonal** is the only cut that moves
material and lighting together — the only path that can reach GT. If the diagonal shows a
hump between the two corners, the fit and GT are *separate basins with a barrier* (local
minimum, not a flat valley); if it descends monotonically, they are one connected valley
the optimiser simply failed to follow. Either way, no axis-aligned sweep can reveal it —
which is why yours looked flat.

In [ ]:
D.show_basins()


## 10. Does VarPro land in the GT funnel? — a second optimiser

Section 9 showed LBFGS parking in a wrong basin behind a tall barrier, with GT at the
bottom of a needle-thin funnel a first-order method has almost no chance of hitting from a
cold start. A Gauss-Newton method like **VarPro** takes large, curvature-aware steps and
can cross ridges LBFGS cannot — so the test is whether its solution already sits *inside*
the GT funnel.

Re-fit the **same** scene with VarPro, keep the **same** worst pixels, and draw each
optimiser's own fit→GT diagonal on one axis. Read it as:

- **VarPro's dot (t=0) already low, its diagonal flat/monotone to GT** → it landed in the
  GT funnel; the barrier LBFGS faced is one VarPro's step size cleared.
- **VarPro's diagonal also has a tall barrier** → it found its *own* separate basin; the
  problem is genuinely multi-modal and even Gauss-Newton needs a warm start (add a
  `curriculum` to `CFG_VP` for the reference GD→VarPro recipe).

`|albedo−GT|` per pixel and the global albedo RMSE quantify how much closer, if at all.

In [ ]:
# GD->VarPro instead of cold VarPro: D.show_varpro(curriculum=[{"optimizer":"LBFGS","n_iter":200}])
D.show_varpro()


## 11. Converged or stalled? — natural vs optimizer-space gradient

The §9 diagonal barrier does **not** prove a local optimum. The real test is the gradient the
optimizer actually descends: `∂loss/∂x`, where `physical = fwd(x)`. Under a **sigmoid** that is
the natural gradient × `p(1-p)`, which **vanishes at a box bound** — so a fit can read
"converged" (`|g_opt|≈0`) while the physical gradient still points toward GT (`|g_nat|>0`).
If instead `|g_opt|` is large, the optimizer merely **stalled** and a GD tail / better settings
would still help.

**VarPro** optimises material in the natural box, so for it `|g_opt| == |g_nat|`: if that is ≈0
it is a genuine local optimum (a GD tail cannot escape it); if not, VarPro stalled. The cell
reports both for the LBFGS/Adam fit and (if §10 ran) the VarPro solution.

In [ ]:
D.optimizer_grad(tag="fit")
if hasattr(D, "est_vp"):
    D.optimizer_grad(est=D.est_vp, sh=D.sh_vp, space="natural", tag="VarPro")


## 12. Which parameter dominates? — one-at-a-time est→GT interpolation

§9's diagonal moves **all** parameters at once, so the sharp GT spike can't be attributed to
any one of them. Here we do the opposite: interpolate **one parameter group est→GT while
holding all others at GT**. At `t=1` every curve hits the full-GT point (the spike); at `t=0`
the curve's height is the loss from **that parameter's estimate alone**. The steepest / highest
curve — and the printed `dominant` column — is the parameter whose error matters most.

In [ ]:
D.show_param_interp()


## 13. Strategies to escape the local optima

The fits land in local optima because the joint **material + lighting** problem is multi-modal
and **gauge-ambiguous** (albedo↔lighting scale, metallic↔F0, roughness↔specular energy). A local
optimizer cannot leave a basin — so the leverage is to **reshape the landscape** or **reseed**.
Grouped by mechanism (each maps to a `cfg` key you can drop into `compare_strategies`):

**A. Fix the gauge / break the ambiguity** — add a term that pins the unobservable direction
- **white-world / mean-albedo gauge** — `lambda_white`: pins the albedo↔lighting scale.
- **metallic binarize** — `lambda_metallic_binarize`: pushes metallic toward 0 or 1, kills metallic↔F0.
- **box penalty (natural space)** — `tr_*="none"` + `lambda_box`: physical range without sigmoid saturation.
- **spatial priors** — `lambda_tv`, `lambda_sparse`: piecewise-smooth material.

**B. Warm-start / continuation** — reach the right basin before the hard problem
- **GD→VarPro** (reference recipe) — a `curriculum` with an Adam phase, then VarPro.
- **specular warmup** — `spec_warmup_steps`: fit diffuse-only first (well-posed), then fade in specular.
- **SH-order homotopy** — per-phase `sh_order` 2→3 (via decompose_scene): smoother lighting first.
- **coarse-to-fine** — downsample curriculum: solve low-res, then refine.

**C. Multi-start / basin hopping** — escape by reseeding
- **random-init ensemble** — `init_seed` / `init_spec_noise_std`, run K times, keep the lowest loss.
- **perturb-and-restart** from the current solution.

**D. Robust loss / reweighting** — soften the sharp minima
- **Huber** — `loss="huber"`: down-weights specular outliers.
- down-weight specular-dominated / low-roughness pixels.

`compare_strategies` runs **one full fit per strategy** (expensive — trim the dict / keep `n_iter`
modest for a screen, then scale the winner up) and ranks them by **intrinsics error, not recon**
(they decouple here). Cross-check the winner with `optimizer_grad()` (did it converge?) and
`show_param_interp()` (did it fix the dominant parameter?).

In [ ]:
# One full fit each — expensive. Trim / lower n_iter for a quick screen.
STRATEGIES = {
    "baseline":          {},
    "white-world gauge": {"lambda_white": 1e-2},
    "TV prior":          {"lambda_tv": 1e-4},
    "metallic binarize": {"lambda_metallic_binarize": 1e-2},
    "spec warmup":       {"spec_warmup_steps": 300},
    "natural + box":     {"tr_albedo": "none", "tr_metallic": "none",
                          "tr_roughness": "none", "lambda_box": 0.1},
    "GD->VarPro":        {"optimizer": "VARPRO", "varpro_space": "natural", "n_iter": 100,
                          "varpro_lam_init": 1e-4, "varpro_n_inner_rho": 10,
                          "curriculum": [{"optimizer": "Adam", "n_iter": 20000, "lr": 0.05}]},
}
D.compare_strategies(STRATEGIES)


## 14. Lighting-focused experiments (§12 says lighting dominates)

§12 showed the recovered **lighting** is the largest error, so the material-side strategies in
§13 can't help much. These two experiments attack the lighting directly (both now wired):

- **`lambda_light_prior`** — pulls the per-image SH toward a reference (`light_prior=<(K,n_sh,3)>`)
  with `||sh−ref||²`, or, with no reference, an **SH-smoothness** prior shrinking the directional
  coefficients. The *anchor* below uses the known reference bank, so it's a near-oracle
  upper bound ("if the lighting were known, is the material then recoverable?"); *smoothness*
  needs no side information.
- **`compare_images`** — refits at increasing image counts. Lighting is **shared** across images,
  so more lightings over-determine it (and each pixel's albedo must explain many lightings —
  photometric stereo). If albedo/lighting error falls with N, under-constrained lighting was the
  culprit and the fix is more observations.

In [ ]:
import numpy as np
REF = np.stack([c.cpu().numpy() for c in D.S["sh_t"]])   # reference lighting (what it was rendered under)
D.compare_strategies({
    "baseline":         {},
    "light smoothness": {"lambda_light_prior": 1e-2},
    "light anchor":     {"lambda_light_prior": 1e-1, "light_prior": REF},   # near-oracle upper bound
    "GD->VarPro":       {"optimizer": "VARPRO", "varpro_space": "natural", "n_iter": 100,
                         "varpro_lam_init": 1e-4, "varpro_n_inner_rho": 10,
                         "curriculum": [{"optimizer": "Adam", "n_iter": 20000, "lr": 0.05}]},
})

# Does MORE IMAGES cure the lighting-dominated error? (one full fit per N — expensive)
D.compare_images([16, 32, 64, 128])


## 15. Oracle: is the material recoverable if the lighting is known?

The two experiments above **refuted** two hypotheses: more images made it *worse* (the per-image
lighting is unknown, so the albedo⊗lighting factorization gauge is structural, not cured by data),
and the λ=0.1 anchor swamped the tiny data loss (~1e-6) and starved the material. The clean test
holds the lighting **fixed at the known reference bank** and optimises material only. If albedo
collapses toward GT, **lighting estimation is the entire problem** and there is no separate
material degeneracy — so the leverage is a lighting prior/estimator, never material regularisers
or more images.

In [ ]:
D.fit_material_given_lighting()   # lighting fixed at the reference; material-only fit


## 16. Sensitivity at GT — how well is each parameter identified at the true optimum?

Sweep each parameter around its **GT** value with all others at GT. Every curve bottoms at the
model floor `r(GT,GT)`; the **width/curvature is the identifiability** — a sharp well is
well-constrained, a flat trough is not. The last two columns probe the lighting: an overall
**scale** (`sh·s`) and the coupled **albedo↔lighting scale** (`albedo·s, sh/s`) — the classic
gauge direction, which is flatter than any single parameter but not perfectly flat (the specular
term and the albedo clamp break the exact cancellation).

Note this is the **local** shape *at* GT. If the wells are all sharp, GT is a clean, well-posed
minimum — which means the failure to reach it is **global** (the optimizer lands in a different
basin), not a local-conditioning problem. That reinforces the §15 oracle: the fix is a better
lighting estimate / basin, not a landscape-reshaping regulariser near GT.

In [ ]:
D.show_gt_sensitivity()


## 17. Research-backed experiments: getting the lighting right

The §15 oracle proved the material is trivially recoverable **given** the lighting, so this is a
**lighting-estimation** problem — exactly what physically-based inverse-rendering work targets.
From nvdiffrec / nvdiffrecmc and the broader literature:

- **Regularisation is essential** to disentangle material and lighting; *smoothness alone is not
  enough*. The workhorse is a **monochrome / white-balance light prior** — assume the lighting is
  mostly achromatic so *colour* is explained by albedo, not baked into the light (`lambda_light_mono`).
- **Staged / progressive optimisation** — estimate the harder block first, then refine. Our
  `fit_lighting_then_material` is the staged form: estimate lighting (GD→VarPro), **freeze** it,
  then fit material only (the realistic version of the oracle — uses the *estimated* lighting).
- **Tractable specular** via split-sum / spherical-gaussian pre-integration — our GGX-SH LUT is the
  analogue. nvdiffrecmc adds Monte-Carlo + denoising for shadows/interreflection (not present in our
  SH forward model).

**Calibrate the regulariser weights to the data-loss scale** (~1e-6 here): a prior of 0.1 swamps the
data and starves the material (we saw this). Judge by intrinsics error and by `fit_material_given_lighting`
recon: if the *estimated-lighting* recon (`GT material under ref`) is high, the estimate is still off.

In [ ]:
import numpy as np

# NB: VarPro solves lighting in closed form and IGNORES lambda_light_prior/mono/box, so the
# prior strategies MUST run under a GD optimiser (LBFGS/Adam) to take effect. And the prior
# weight has to be scaled to the data-loss magnitude of THIS scene (a fixed value won't transfer),
# so the anchor is swept. GD->VarPro is the joint-optimiser reference.
GD  = {"optimizer": "LBFGS", "n_iter": 300, "lbfgs_max_iter": 40}
REF = np.stack([c.cpu().numpy() for c in D.S["sh_t"]])   # known reference bank (near-oracle anchor)
D.compare_strategies({
    "baseline (GD)":     {**GD},
    "light monochrome":  {**GD, "lambda_light_mono": 1e-3},
    "light anchor 1e-4": {**GD, "lambda_light_prior": 1e-4, "light_prior": REF},
    "light anchor 1e-2": {**GD, "lambda_light_prior": 1e-2, "light_prior": REF},
    "light anchor 1e0":  {**GD, "lambda_light_prior": 1e0,  "light_prior": REF},
    "GD->VarPro":        {"optimizer": "VARPRO", "varpro_space": "natural", "n_iter": 100,
                         "varpro_lam_init": 1e-4, "varpro_n_inner_rho": 10,
                         "curriculum": [{"optimizer": "Adam", "n_iter": 20000, "lr": 0.05}]},
})

# Staged pipeline (nvdiffrec-style; your idea): estimate lighting -> freeze -> material-only.
D.fit_lighting_then_material()


## 18. Coarse-to-fine over colour: grayscale first, then split colour

Fit a **grayscale/monochrome** decomposition first (luminance observations + strong monochrome-light
prior) — a low-DOF problem meant to estimate the lighting *shape* in a better-conditioned landscape —
then recover colour, warm-started from it (`freeze_light=False`, mild monochrome so the shape does not
re-drift) or with the grayscale light held fixed (`freeze_light=True`, the white-light split).

**Watch stage 1's `lighting-shape RMSE`.** Dropping the colour DOF shrinks the landscape but adds no
constraints, so the grayscale fit can *still* land a wrong lighting basin — and then colour inherits it
(on the synthetic scene it did: stage-1 lighting ≈0.31, no gain). It only pays off if the grayscale
stage actually finds the lighting; pair it with a warm start / GD→VarPro on stage 1 (`gray_cfg=...`),
or an anchor, if stage-1 RMSE is high.

In [ ]:
# grayscale -> colour. Warm-start the RGB lighting from the grayscale basin (or freeze it).
D.fit_grayscale_then_color(freeze_light=False)
# D.fit_grayscale_then_color(freeze_light=True)                         # white-light split (monochrome ceiling)
# D.fit_grayscale_then_color(gray_cfg={"optimizer":"LBFGS","n_iter":500})  # give stage 1 a stronger fit
